# Utils · Common Functions
### Shared helper functions used across all notebooks
###  Always loaded via `%run ../utils/common_functions`

In [0]:
%run ../config/pipeline_config

In [0]:
# ── Imports ──────────────────────────────────────────────────────────
from datetime import datetime, date,timedelta
from pyspark.sql import functions as F
from pyspark.sql.types import *
import uuid
import requests

In [0]:
# ── Pipeline Logger ──────────────────────────────────────────────────

def log(step,message,status="INFO"):
    ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    print(f"[{ts}] [{status}] [{step}] {message}")



In [0]:
def get_watermark(source_name,location_id):
    """Read last loaded date for a given source and location."""

    df = spark.sql(f"""
                   SELECT last_loaded_date
                   FROM {TBL_WATERMARK}
                   WHERE source_name = '{source_name}'
                   AND location_id = {location_id}
                   """
    )
    if df.count() ==0:
        log("WATERMARK", f"No watermark found for {source_name} / location {location_id} — using default start date")
        return DEFAULT_START_DATE
    
    last_date = df.collect()[0]['last_loaded_date']
    log("WATERMARK", f"Found watermark for {source_name} / location {location_id} → {last_date}")
    return str(last_date)






def update_watermark(source_name,location_id,last_loaded_date):
        """Update watermark after successful load."""
        spark.sql(f"""
                  MERGE INTO {TBL_WATERMARK} AS target
                  USING (
                       SELECT 
                            '{source_name}' as source_name,
                            {location_id}   as location_id,
                            cast('{last_loaded_date}' as DATE) AS last_loaded_date,
                            current_timestamp()       as updated_at

                  ) as source
                  ON target.source_name = source.source_name
                  AND target.location_id = source.location_id
                  WHEN MATCHED THEN UPDATE SET *
                  WHEN NOT MATCHED THEN INSERT *
                  """)

        log("WATERMARK", f"Updated watermark for {source_name} / location {location_id} → {last_loaded_date}")


In [0]:
# ── DQ Results Writer ────────────────────────────────────────────────

# ── DQ Results Writer ────────────────────────────────────────────────

def write_dq_result(run_id,source_table,check_name,records_checked,records_failed,details=""):
        """Write a single DQ check result to the dq_results log table."""
        failure_rate = round((records_failed / records_checked) * 100,2) \
            if records_checked > 0 else 0.0

        status = "PASS" if records_failed==0 else "FAIL"
        log("DQ", f"{check_name} → {status} "
              f"({records_failed}/{records_checked} failed, {failure_rate}%)",
        status=status)
        row =[(
            run_id,
            datetime.now(),
            source_table,
            check_name,
            status,
            records_checked,
            records_failed,
            failure_rate,
            details
        )]

        schema = StructType([
            StructField("run_id",           StringType(),  True),
            StructField("run_ts",           TimestampType(),True),
            StructField("source_table",     StringType(),  True),
            StructField("check_name",       StringType(),  True),
            StructField("status",           StringType(),  True),
            StructField("records_checked",  LongType(),    True),
            StructField("records_failed",   LongType(),    True),
            StructField("failure_rate_pct", DoubleType(),  True),
            StructField("details",          StringType(),  True),

        ])
        df = spark.createDataFrame(row,schema=schema)
        df.write.format("delta").mode("append").saveAsTable(TBL_DQ_RESULTS)




In [0]:
def get_load_date_range(source_name,location_id):
    """
    Calculate start and end dates for this incremental run.
    Reads watermark → adds 1 day → fetches next BATCH_DAYS days.
    """
    last_date = get_watermark(source_name,location_id)
    start_date   = str(date.fromisoformat(last_date) + timedelta(days=1))
    end_date =   str(date.fromisoformat(start_date) + timedelta(days=BATCH_DAYS-1))
    # Never go beyond yesterday (API only has historical data)

    yesterday    = str(date.today() - timedelta(days=1))
    if end_date > yesterday:
        end_date = yesterday
    log("DATE_RANGE", f"source={source_name} | location={location_id} "
                      f"| start={start_date} | end={end_date}")
    return start_date, end_date
